# 🛰️ A rover learning on an exoplanet — watching the Q-values grow

This notebook is a companion to the classroom lesson. There, you watched your micro:bit + Cutebot
learn to cross the 3×3 grid, but the Q-table itself was invisible — hidden inside the micro:bit's memory.

**Here, you can see it.** This notebook runs the exact same environment and the exact same Q-learning
rule as the code you flashed to your micro:bit, but after every single episode it saves a snapshot of
the whole Q-table. You can then scrub through those snapshots and watch the numbers grow in front of you
— and watch the reward from the goal *propagate backwards*, cell by cell, exactly like we discussed in
Step 5 of the lesson.

**How to use this notebook:** click **Runtime → Run all** (or run the cells one by one from top to
bottom), then play with the sliders and try the "tune it yourself" section at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, HTML

## 1. The environment — the same 3×3 grid as your rover

Same convention as the micro:bit code:

- 9 states, numbered 0–8. `state = row*3 + col`, with row 0 at the **top** (state 0 = start,
  top-left) and state 8 = the goal (bottom-right, the research station).
- 4 actions: `0 = UP`, `1 = DOWN`, `2 = LEFT`, `3 = RIGHT`.
- Rewards: **-1** for a normal move, **-5** for trying to drive off the grid (the rover stays put),
  **+10** for reaching the goal.

In [ ]:
N_STATES = 9
N_ACTIONS = 4                      # 0=UP  1=DOWN  2=LEFT  3=RIGHT
ACTION_NAMES = ["UP", "DOWN", "LEFT", "RIGHT"]
ACTION_DXY = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}  # (dx, dy) on the grid
GOAL = 8

def state_to_xy(state):
    return state % 3, state // 3

def xy_to_state(x, y):
    return y * 3 + x

def step(state, action):
    """Exactly the logic of get_next_state() in RL_leds-3x3.py."""
    x, y = state_to_xy(state)
    dx, dy = ACTION_DXY[action]
    nx, ny = x + dx, y + dy
    if nx < 0 or nx > 2 or ny < 0 or ny > 2:
        return state, -5, False               # drove off the grid: stay, penalty
    ns = xy_to_state(nx, ny)
    if ns == GOAL:
        return ns, 10, True                    # reached the goal!
    return ns, -1, False                       # normal move

## 2. The Q-learning agent

Same rule as Step 5 of the lesson:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \,\Big]$$

with the same parameters as the code on your micro:bit: `alpha = 0.5`, `gamma = 0.9`,
`epsilon = 0.3`. Because the grid is tiny, learning is fast — a handful of episodes is
already enough to see the pattern, and by ~30-50 episodes it has essentially converged.

Feel free to change the numbers in the next cell later (there's a whole section on that below) —
for now, just run it as-is.

In [ ]:
ALPHA = 0.5          # learning rate
GAMMA = 0.9           # discount factor
EPSILON = 0.3         # exploration probability
N_EPISODES = 150      # how many episodes to train for
MAX_STEPS = 50        # safety cap per episode
SEED = 7              # change this for a different random run

rng = np.random.default_rng(SEED)

In [ ]:
Q = np.zeros((N_STATES, N_ACTIONS))

# a snapshot of the whole Q-table after every episode, so we can scrub through history
Q_history = np.zeros((N_EPISODES + 1, N_STATES, N_ACTIONS))
episode_lengths = []

for ep in range(N_EPISODES):
    state = 0
    steps = 0
    done = False
    while not done and steps < MAX_STEPS:
        # choose an action: explore with probability epsilon, else exploit
        if rng.random() < EPSILON:
            action = rng.integers(0, N_ACTIONS)
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, done = step(state, action)

        # the Q-learning update -- identical to the line in RL_leds-3x3.py
        Q[state, action] += ALPHA * (reward + GAMMA * np.max(Q[next_state]) - Q[state, action])

        state = next_state
        steps += 1

    episode_lengths.append(steps)
    Q_history[ep + 1] = Q.copy()

print(f"Trained for {N_EPISODES} episodes.")
print(f"Steps in episode 1: {episode_lengths[0]}   |   steps in last episode: {episode_lengths[-1]}")
print("\nFinal Q-table (rows = states 0-8, columns = UP, DOWN, LEFT, RIGHT):")
print(np.round(Q, 2))

## 3. Watch the Q-values grow

The plot below draws the grid exactly like your printed handout: each cell shows its
**best** Q-value (the highest of its 4 action-values) as both a color and a number, and an
arrow for the action the rover currently believes is best from that cell. The goal cell is
always shown in orange, since the rover never *acts* from the goal (the episode just ends there).

Darker green = higher Q-value = "the rover is more confident this cell is a good place to be
on the way to the goal."

In [ ]:
def plot_grid(Q_snapshot, episode, ax=None, vmax=10):
    """Draws the 3x3 grid: color + number = best Q-value, arrow = best action."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.clear()

    best_value = Q_snapshot.max(axis=1)
    best_action = Q_snapshot.argmax(axis=1)

    for s in range(N_STATES):
        x, y = state_to_xy(s)
        py = 2 - y   # flip so row 0 (top of the grid) is drawn at the top of the plot

        if s == GOAL:
            ax.add_patch(plt.Rectangle((x, py), 1, 1, facecolor="#E76F51", edgecolor="#264653", lw=1.5))
            ax.text(x + 0.5, py + 0.5, "GOAL", ha="center", va="center",
                    color="white", fontsize=12, fontweight="bold")
            continue

        val = best_value[s]
        t = np.clip(val / vmax, 0, 1)
        color = matplotlib.colormaps["YlGn"](0.15 + 0.75 * t)
        ax.add_patch(plt.Rectangle((x, py), 1, 1, facecolor=color, edgecolor="#264653", lw=1.5))
        ax.text(x + 0.08, py + 0.86, f"s{s}", fontsize=8, color="#264653")
        ax.text(x + 0.5, py + 0.28, f"{val:0.1f}", ha="center", va="center",
                fontsize=14, fontweight="bold", color="#1b2e23")

        if val > 0.05:  # only draw an arrow once the rover has actually learned something here
            gdx, gdy = ACTION_DXY[best_action[s]]
            pdx, pdy = 0.32 * gdx, -0.32 * gdy   # flip dy: py = 2 - y
            ax.annotate("", xy=(x + 0.5 + pdx, py + 0.5 + pdy), xytext=(x + 0.5, py + 0.5),
                        arrowprops=dict(arrowstyle="-|>", color="#264653", lw=2.2))

    ax.set_xlim(0, 3); ax.set_ylim(0, 3)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal")
    ax.set_title(f"Episode {episode}", fontsize=13)
    return ax


# a quick look at 4 moments in training
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
for ax, ep in zip(axes, [0, 5, 20, N_EPISODES]):
    plot_grid(Q_history[ep], ep, ax=ax)
plt.tight_layout()
plt.show()

### Scrub through training yourself

Drag the slider below, episode by episode, and watch the numbers and arrows change.
Try going slowly from episode 0 to about episode 30 -- that's where almost everything happens.

In [ ]:
import ipywidgets as widgets

def _show(episode):
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    plot_grid(Q_history[episode], episode, ax=ax)
    plt.show()

widgets.interact(
    _show,
    episode=widgets.IntSlider(min=0, max=N_EPISODES, step=1, value=0,
                               description="Episode", continuous_update=True,
                               layout=widgets.Layout(width="520px")),
);

## 4. Reward propagating backwards

Here is the exact question from the handout, answered visually: *once the rover reaches the goal
for the first time, how does that +10 reward "travel backwards" to help it earlier in the grid?*

Below we track the Q-value of one action, for four cells that sit along the shortest path to the
goal (`0 → 1 → 2 → 5 → 8`), from the one right next to the goal to the one furthest away:

- **s5 → DOWN (into the goal)** — one step away from the goal
- **s2 → DOWN** — two steps away
- **s1 → RIGHT** — three steps away
- **s0 → RIGHT** — four steps away (the start!)

Watch which curve rises first.

In [ ]:
path = [
    (5, 1, "s5 -> DOWN  (1 step from goal)"),
    (2, 1, "s2 -> DOWN  (2 steps from goal)"),
    (1, 3, "s1 -> RIGHT (3 steps from goal)"),
    (0, 3, "s0 -> RIGHT (4 steps from goal, the start)"),
]
colors = ["#E76F51", "#F0A583", "#4FB0A5", "#2A9D8F"]

fig, ax = plt.subplots(figsize=(8, 4.6))
for (s, a, label), c in zip(path, colors):
    ax.plot(Q_history[:, s, a], label=label, color=c, lw=2.6)

ax.set_xlabel("Episode")
ax.set_ylabel("Q-value")
ax.set_title("The reward from the goal propagates backwards, cell by cell")
ax.legend(loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Notice the order in which the curves take off: the cell closest to the goal (s5) shoots up")
print("almost immediately, while the starting cell (s0) only starts climbing once s1 and s2 already")
print("have a meaningful Q-value to 'borrow' from -- exactly the gamma * max(Q(s', a')) term at work.")

## 5. Bonus: is the rover actually getting faster?

Another way to see learning happening: how many steps does each episode take? Early on, with
an empty Q-table, the rover wanders (and sometimes drives into edges). Later, it should settle
close to 4 steps -- the length of the shortest path from start to goal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(episode_lengths, color="#2A9D8F", alpha=0.4, label="steps in this episode")

# a rolling average makes the trend easier to see through the exploration noise
window = 5
if len(episode_lengths) >= window:
    roll = np.convolve(episode_lengths, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(episode_lengths)), roll, color="#264653", lw=2.5,
            label=f"{window}-episode rolling average")

ax.axhline(4, color="#E76F51", ls="--", lw=1.5, label="shortest possible path (4 steps)")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps to reach the goal")
ax.set_title("Episode length over training")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Try it yourself

Go back up to the **parameters cell** in Section 2, change one thing, then re-run every cell
below it (or just click **Runtime → Run all**). Some ideas to try, one at a time:

- **`EPSILON = 0`** — the rover never explores, only exploits. What happens? (Hint: with an
  empty Q-table and no exploration, it might get stuck taking the same bad action forever.)
- **`EPSILON = 1`** — the rover always picks a random action. Does it ever *stop* looking random
  in the grid plot?
- **`ALPHA = 0.05`** (small learning rate) vs **`ALPHA = 1.0`** (large) — which one propagates the
  reward backwards faster? Which one looks "shakier" once it has learned?
- **`GAMMA = 0`** — the rover only cares about the *immediate* reward, never the future. Look at
  the propagation plot in Section 4: does the reward still travel backwards at all?
- **`SEED`** — change this to any other number to see a different random training run.

Discuss with your partner: which parameter had the biggest effect on *how fast* the green
color spread across the grid?

## Bonus: an animation of the whole training run

Run the cell below to render every episode as a short animation instead of scrubbing by hand
— useful if you want to project it for the whole class.

In [ ]:
from matplotlib import animation

frames = list(range(0, N_EPISODES + 1, max(1, N_EPISODES // 60)))  # ~60 frames, evenly spaced
fig, ax = plt.subplots(figsize=(4.8, 4.8))

def _update(i):
    plot_grid(Q_history[frames[i]], frames[i], ax=ax)
    return []

anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=150)
plt.close(fig)
HTML(anim.to_jshtml())